# Test the Spark feature job on a sample of HI-Small

Run this notebook first. It must end with **PARITY OK** before you run `spark_notebook.ipynb`.

1. Make a 1-in-12 sample of HI-Small (in S3) and upload it.
2. Run the Spark feature job on the sample (SageMaker Processing, PySpark).
3. Run the parity check: the pandas build of the same sample against the Spark output. Every feature, label and key must match.
4. Write a PASS or FAIL verdict to S3. `spark_notebook.ipynb` reads it before the HI-Large run.

The Spark job and the pandas code are tested together here, on the same file.

## 1. Configuration

In [ ]:
from pathlib import Path
import shutil

import boto3
import sagemaker
from sagemaker.session import Session

ROLE_ARN = sagemaker.get_execution_role()  # the Studio execution role; nothing hard-coded
REGION = "ap-south-1"
BUCKET = "neu-datasets"
NB_DIR = Path.cwd()  # run from the notebooks/ folder
WORK = NB_DIR / "sagemaker_work"
JOB = WORK / "job"  # everything the SageMaker jobs need, bundled here
sess = Session(boto_session=boto3.Session(region_name=REGION))
s3 = boto3.client("s3", region_name=REGION)


def build_job_folder():
    """Copies the Spark job, the rule module, the parity check and triage_lib into one folder for SageMaker."""
    shutil.rmtree(JOB, ignore_errors=True)
    JOB.mkdir(parents=True)
    for f in (NB_DIR / "spark").glob("*.py"):
        shutil.copy(f, JOB)
    shutil.copytree(NB_DIR / "triage_lib", JOB / "triage_lib", ignore=shutil.ignore_patterns("__pycache__"))
    (JOB / "requirements.txt").write_text("pandas==3.0.6\nnumpy==2.5.4\npyarrow==26.0.0\n", encoding="utf-8")
    print("job folder:", sorted(p.name for p in JOB.iterdir()))

HORIZON_SAMPLE = 2       # HI-Small is about 10 active days, so horizon 2 (HI-Large uses 7)
SAMPLE_EVERY = 12        # keep every 12th line (plus the header)
SAMPLE_KEY = "samples/HI-Small_sample12.csv"
VERDICT_KEY = "parity/sample/verdict/verdict.txt"

## 2. Make the sample from HI-Small in S3 and upload it

The sample is about 1/12 of HI-Small, all days kept. It is uploaded once, and re-running this cell reuses the existing copy.

In [ ]:
WORK.mkdir(exist_ok=True)
full = WORK / "HI-Small_Trans.csv"
if not full.exists():
    s3.download_file(BUCKET, "HI-Small_Trans.csv", str(full))
sample = WORK / "HI-Small_sample12.csv"
with full.open() as src, sample.open("w") as dst:
    for n, line in enumerate(src, start=1):  # n = 1 is the header
        if n == 1 or n % SAMPLE_EVERY == 0:
            dst.write(line)
print(f"sample: {sum(1 for _ in sample.open()) - 1:,} transactions")

try:
    s3.head_object(Bucket=BUCKET, Key=SAMPLE_KEY)
    print(f"already in S3: s3://{BUCKET}/{SAMPLE_KEY}")
except s3.exceptions.ClientError:
    s3.upload_file(str(sample), BUCKET, SAMPLE_KEY)
    print(f"uploaded s3://{BUCKET}/{SAMPLE_KEY}")

## 3. Bundle the jobs

The Spark job, the rule module, the parity check and `triage_lib` are copied into one folder. Requirements for the parity job are in `requirements.txt`.

In [ ]:
build_job_folder()

## 4. Spark feature job on the sample (SageMaker Processing)

In [ ]:
from sagemaker.spark.processing import PySparkProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

spark = PySparkProcessor(
    base_job_name="triage-spark-sample", framework_version="3.5", role=ROLE_ARN,
    instance_count=2, instance_type="ml.m5.2xlarge", max_runtime_in_seconds=3600, sagemaker_session=sess,
)
spark.run(
    submit_app=str(JOB / "build_features_spark.py"),
    submit_py_files=[str(JOB / "rules_r1_r4.py")],
    arguments=[
        "--input", "/opt/ml/processing/input/csv/HI-Small_sample12.csv",
        "--output", "/opt/ml/processing/output/features",
        "--manifest", "/opt/ml/processing/output/manifest/features_manifest.json",
        "--horizon", str(HORIZON_SAMPLE), "--partitions", "16",
    ],
    inputs=[ProcessingInput(source=f"s3://{BUCKET}/{SAMPLE_KEY}", destination="/opt/ml/processing/input/csv")],
    outputs=[
        ProcessingOutput(source="/opt/ml/processing/output/features", destination=f"s3://{BUCKET}/parity/sample/features"),
        ProcessingOutput(source="/opt/ml/processing/output/manifest", destination=f"s3://{BUCKET}/parity/sample/manifest"),
    ],
)
print("Spark sample job finished:", spark.latest_job.job_name)

## 4b. If a job failed: read the cluster error

Only needed if section 4 or 5 raised an error. `show_errors` prints the error lines from the driver log of that job.

In [ ]:
import boto3


def show_errors(job_name, max_lines=40):
    """Prints error lines from the cluster driver log of a SageMaker processing job."""
    logs = boto3.client("logs", region_name=REGION)
    group = "/aws/sagemaker/ProcessingJobs"
    streams = logs.describe_log_streams(logGroupName=group, logStreamNamePrefix=job_name)["logStreams"]
    for s in streams:
        name = s["logStreamName"]
        if "algo-1" not in name:
            continue
        hits, token = [], None
        while True:
            kwargs = dict(logGroupName=group, logStreamName=name, startFromHead=True)
            if token:
                kwargs["nextToken"] = token
            resp = logs.get_log_events(**kwargs)
            for e in resp["events"]:
                m = e["message"]
                if "#011" in m:  # skip Java stack-trace frames
                    continue
                if any(k in m for k in ("Traceback", "Error:", "Exception:", "ERROR", "OutOfMemory", "Killed")):
                    hits.append(m[:300])
            if not resp["events"] or resp["nextForwardToken"] == token:
                break
            token = resp["nextForwardToken"]
        print(name, "matches:", len(hits))
        print("\n".join(hits[:max_lines]))


# If section 4 failed, run: show_errors(spark.latest_job.job_name)

## 5. Parity check: pandas build of the sample vs the Spark output

This runs `parity_check.py` in a scikit-learn processing container. It fails the job if any feature, label or key differs. The verdict is written to S3 either way.

In [ ]:
from sagemaker.sklearn.processing import SKLearnProcessor

parity = SKLearnProcessor(
    framework_version="1.2-1", role=ROLE_ARN, instance_type="ml.m5.2xlarge", instance_count=1,
    base_job_name="triage-parity", max_runtime_in_seconds=3600, sagemaker_session=sess,
)
try:
    parity.run(
        code="parity_check.py", source_dir=str(JOB),
        arguments=[
            "--csv", "/opt/ml/processing/input/csv/HI-Small_sample12.csv",
            "--spark-dir", "/opt/ml/processing/input/spark/features",
            "--manifest", "/opt/ml/processing/input/manifest/features_manifest.json",
            "--horizon", str(HORIZON_SAMPLE),
            "--verdict", "/opt/ml/processing/output/verdict/verdict.txt",
        ],
        inputs=[
            ProcessingInput(source=f"s3://{BUCKET}/{SAMPLE_KEY}", destination="/opt/ml/processing/input/csv"),
            ProcessingInput(source=f"s3://{BUCKET}/parity/sample/features", destination="/opt/ml/processing/input/spark/features"),
            ProcessingInput(source=f"s3://{BUCKET}/parity/sample/manifest", destination="/opt/ml/processing/input/manifest"),
        ],
        outputs=[ProcessingOutput(source="/opt/ml/processing/output/verdict", destination=f"s3://{BUCKET}/parity/sample/verdict")],
    )
except Exception as exc:  # a FAIL verdict makes the job fail; the verdict file still tells you why
    print("parity job did not finish cleanly:", exc.__class__.__name__)
    show_errors(parity.latest_job.job_name)

## 6. Verdict

`PASS` means `spark_notebook.ipynb` may run. `FAIL` lists the columns that differ. Fix the Spark job, then re-run from section 4.

In [ ]:
try:
    verdict = s3.get_object(Bucket=BUCKET, Key=VERDICT_KEY)["Body"].read().decode("utf-8")
except s3.exceptions.NoSuchKey:
    verdict = "FAIL\nno verdict written: the parity job did not finish. See section 4b for the error."
print(verdict)
if verdict.startswith("PASS"):
    print("PARITY OK: run spark_notebook.ipynb next.")
else:
    print("PARITY NOT OK: do not run spark_notebook.ipynb.")